# EDA — Predicción de Churn en Telecomunicaciones

**Asignatura:** Machine Learning  
**Dataset:** IBM Telco Customer Churn (7,043 clientes)  
**Objetivo:** Explorar la estructura del dataset, detectar patrones y formular hipótesis para el modelado.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

Se importan las librerías necesarias. `pandas` y `numpy` manejan los datos en formato tabular, `matplotlib` y `seaborn` generan las visualizaciones, y `pathlib` construye rutas de archivo compatibles entre sistemas operativos. Las opciones de display se configuran para ver todas las columnas y hasta 100 filas sin truncamiento.

### 2. Carga del Dataset

Se carga el dataset original desde la carpeta `data/raw/`. La ruta se construye de forma relativa
al directorio del proyecto usando `pathlib.Path`, lo que garantiza compatibilidad entre sistemas operativos.

Se espera observar el dataset con 7,043 registros y 21 columnas, incluyendo variables demográficas,
servicios contratados, información contractual y la variable objetivo `Churn`.


In [ ]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = BASE_DIR / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

df.head()

El dataset se carga con 7.043 filas y 21 columnas. Se pueden ver variables demográficas como `gender` y `SeniorCitizen`, servicios contratados como `InternetService` y `OnlineSecurity`, variables de facturación, y la variable objetivo `Churn` con valores `'Yes'` y `'No'`.

In [ ]:
df.shape

El dataset tiene 7.043 registros y 21 columnas.

### 3. Calidad del Dataset

Se analiza la calidad del dataset mediante tres verificaciones fundamentales:
- **Dimensiones** (`shape`): número de filas y columnas.
- **Tipos de datos** (`info`): permite identificar columnas que requieren conversión de tipo.
- **Estadísticas descriptivas** (`describe`): distribución de variables numéricas y frecuencias de categóricas.
- **Valores faltantes** (`isna().sum()`): identifica columnas con datos incompletos.
- **Registros duplicados** (`duplicated().sum()`): verifica la unicidad de los registros.

Se anticipa que `TotalCharges`, aunque debería ser numérica, puede estar codificada como `object`,
lo que genera valores nulos al convertirla.


In [ ]:
df.info()

La mayoría de columnas son tipo `object`. `SeniorCitizen` y `tenure` son enteros y `MonthlyCharges` es decimal. El detalle importante es que `TotalCharges` aparece como `object` en lugar de numérica, lo que indica que tiene algún valor que pandas no puede interpretar como número.

In [ ]:
df.describe(include="all").T

`tenure` va de 0 a 72 meses con media de 32, y `MonthlyCharges` de $18.25 a $118.75. `TotalCharges` tiene 6.531 valores únicos y aparece como `object`, confirmando que pandas no la está tratando como número a pesar de contener precios.

In [ ]:
df.columns.tolist()

Se listan las 21 columnas del dataset. `customerID` es un identificador único que no aporta valor predictivo y se eliminará antes del modelado.

In [ ]:
df.isna().sum().sort_values(ascending=False)

No aparece ningún nulo, pero esto es engañoso. `TotalCharges` contiene cadenas vacías que pandas no detecta como `NaN` hasta que se intenta convertirlas a número.

In [ ]:
df.duplicated().sum()

No hay registros duplicados en el dataset. Cada fila corresponde a un cliente distinto.

In [ ]:
df["customerID"].duplicated().sum()

Todos los `customerID` son únicos. Esta columna se puede eliminar con confianza antes del modelado.

In [ ]:
df["Churn"].value_counts()

Hay 5.174 clientes que no hicieron churn y 1.869 que sí. Las clases están desbalanceadas, con la clase negativa triplicando a la positiva.

### 4. Análisis de la Variable Objetivo

Se analiza la distribución de la variable `Churn` para cuantificar el desbalance entre clases.
Este paso es crítico porque un desbalance significativo puede hacer que modelos entrenados con
`accuracy` como única métrica sean engañosos: un clasificador que siempre predice "No Churn"
alcanzaría ~73% de accuracy sin ningún valor predictivo real.

Se espera confirmar que existe un desbalance moderado (~73% No / ~27% Sí), lo que justifica
el uso de métricas como AUC-ROC, F1-score y Recall como criterios primarios de evaluación.


In [ ]:
df["Churn"].value_counts(normalize=True) * 100

El 73.46% de los clientes no abandonó y el 26.54% sí. Un modelo que siempre prediga "No Churn" tendría 73% de accuracy sin ningún poder predictivo real, por eso se usarán métricas como AUC-ROC y F1-score.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Distribución de la variable objetivo: Churn")
plt.xlabel("Churn")
plt.ylabel("Cantidad de clientes")
plt.show()

La barra de "No" es casi tres veces más alta que la de "Yes". La clase mayoritaria domina claramente, confirmando el desbalance.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

Al convertir con `errors='coerce'`, las cadenas vacías pasan a ser `NaN`. Esto permite identificar exactamente cuántos registros tienen ese problema.

In [ ]:
df["TotalCharges"].isna().sum()

Aparecen 11 valores nulos en `TotalCharges`, todos correspondientes a clientes con `tenure = 0`.

In [ ]:
df[df["TotalCharges"].isna()]

Los 11 clientes con `TotalCharges` nulo tienen todos `tenure = 0`, es decir, son clientes nuevos que aún no han completado su primer mes. Todos tienen `Churn = 'No'`, lo cual es lógico: no pueden haber abandonado si acaban de entrar. El valor correcto para imputar es 0.

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

Los 11 nulos se reemplazan con 0, representando que estos clientes no tienen cargos totales acumulados todavía.

In [ ]:
df["TotalCharges"].isna().sum()

Ningún valor nulo en `TotalCharges`. La corrección quedó aplicada correctamente.

In [ ]:
target = "Churn"

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

categorical_cols = [col for col in categorical_cols if col not in ["customerID", target]]

print("Variables numéricas:")
print(numeric_cols)

print("\nVariables categóricas:")
print(categorical_cols)

Se separan 4 variables numéricas (`SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges`) y 15 categóricas. `customerID` y `Churn` quedan fuera: el primero es identificador y el segundo es la variable objetivo.

In [ ]:
df[numeric_cols].describe().T

`SeniorCitizen` es prácticamente binaria con solo un 16.2% de adultos mayores. `tenure` tiene alta dispersión —de 0 a 72 meses— con mediana en 29. `MonthlyCharges` va de $18.25 a $118.75. `TotalCharges` tiene aún más variabilidad, lo que justifica escalar las variables antes de entrenar los modelos.

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df, x=col, hue="Churn", kde=True, bins=30)
    plt.title(f"Distribución de {col} según Churn")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.show()

Los histogramas muestran que los clientes con churn tienden a tener menor `tenure` —la mayoría se concentra en los primeros meses—, `MonthlyCharges` más altos, y `TotalCharges` menores por haber acumulado menos tiempo de facturación. Las diferencias en la distribución entre ambas clases son visibles en las tres variables.

In [ ]:
for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].value_counts())

El contrato mensual es el más frecuente con 3.875 clientes. La fibra óptica es el servicio de internet predominante. En servicios adicionales como `OnlineSecurity` y `TechSupport`, la categoría "No" supera ampliamente a "Yes", lo que indica que muchos clientes no los tienen contratados. El pago por cheque electrónico es el método más usado con 2.365 clientes.

In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=col, hue="Churn")
    plt.title(f"{col} según Churn")
    plt.xlabel(col)
    plt.ylabel("Cantidad de clientes")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

Se ve que dentro de `Contract Month-to-month` la barra naranja (churn) es proporcionalmente mucho más alta que en One year o Two year. Para `InternetService`, la fibra óptica tiene claramente más churn que DSL. En `OnlineSecurity` y `TechSupport`, quienes no tienen el servicio muestran una proporción de churn visiblemente mayor.

In [ ]:
def churn_rate_by_category(data, column):
    resumen = (
        data.groupby(column)["Churn"]
        .value_counts(normalize=True)
        .rename("proporcion")
        .reset_index()
    )
    
    resumen["porcentaje"] = resumen["proporcion"] * 100
    return resumen

churn_rate_by_category(df, "Contract")

Para `Contract`, el contrato mensual tiene un 42.71% de churn, el anual un 11.27% y el bianual apenas un 2.83%. Es una de las diferencias más grandes entre categorías de todo el dataset.

In [ ]:
for col in ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity", "TechSupport", "PaperlessBilling"]:
    print(f"\nTasa de churn por {col}")
    display(churn_rate_by_category(df, col))

Los patrones más llamativos: el pago por cheque electrónico tiene la tasa de churn más alta con 45.29%, seguido del contrato mensual con 42.71%. No tener `OnlineSecurity` ni `TechSupport` está asociado a tasas superiores al 41%. Los clientes con fibra óptica tienen 41.89% de churn, mientras que los que no tienen internet tienen apenas 7.40%.

## Diccionario de variables

Antes de iniciar el análisis exploratorio, se presenta un diccionario de las principales variables del dataset. Esto permite comprender el significado de cada atributo y su posible relación con la variable objetivo `Churn`.

In [ ]:
diccionario_variables = pd.DataFrame({
    "Variable": [
        "customerID", "gender", "SeniorCitizen", "Partner", "Dependents",
        "tenure", "PhoneService", "MultipleLines", "InternetService",
        "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
        "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling",
        "PaymentMethod", "MonthlyCharges", "TotalCharges", "Churn"
    ],
    "Descripción": [
        "Identificador único del cliente.",
        "Género del cliente.",
        "Indica si el cliente es adulto mayor: 1 = Sí, 0 = No.",
        "Indica si el cliente tiene pareja.",
        "Indica si el cliente tiene dependientes.",
        "Número de meses que el cliente ha permanecido con la empresa.",
        "Indica si el cliente tiene servicio telefónico.",
        "Indica si el cliente tiene múltiples líneas telefónicas.",
        "Tipo de servicio de internet contratado.",
        "Indica si el cliente tiene seguridad en línea.",
        "Indica si el cliente tiene respaldo en línea.",
        "Indica si el cliente tiene protección de dispositivos.",
        "Indica si el cliente tiene soporte técnico.",
        "Indica si el cliente tiene servicio de televisión por streaming.",
        "Indica si el cliente tiene servicio de películas por streaming.",
        "Tipo de contrato del cliente.",
        "Indica si el cliente usa facturación electrónica.",
        "Método de pago utilizado por el cliente.",
        "Cargo mensual del cliente.",
        "Cargo total acumulado del cliente.",
        "Variable objetivo: indica si el cliente abandonó la empresa."
    ],
    "Tipo esperado": [
        "Categórica identificadora", "Categórica", "Numérica binaria", 
        "Categórica", "Categórica", "Numérica", "Categórica", 
        "Categórica", "Categórica", "Categórica", "Categórica", 
        "Categórica", "Categórica", "Categórica", "Categórica", 
        "Categórica", "Categórica", "Categórica", "Numérica", 
        "Numérica", "Categórica objetivo"
    ]
})

diccionario_variables

El diccionario presenta las 21 variables con su descripción y tipo esperado. Se puede ver qué variables son numéricas, cuáles categóricas y cuál es la variable objetivo.

### 5. Análisis de Variables Numéricas

Se analiza la distribución de las cuatro variables numéricas del dataset (`SeniorCitizen`, `tenure`,
`MonthlyCharges`, `TotalCharges`) segmentadas por la variable objetivo `Churn`.

Se utilizan `boxplots` y `histogramas` para visualizar:
- La distribución de cada variable y sus percentiles.
- Las diferencias estadísticas entre clientes con y sin churn.
- La presencia de valores atípicos que pudieran afectar al escalado.

Se anticipan diferencias significativas en `tenure` (antigüedad) y `MonthlyCharges` (cargos mensuales)
entre las dos clases de la variable objetivo.


In [ ]:
numeric_analysis_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in numeric_analysis_cols:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x="Churn", y=col)
    
    plt.title(f"Distribución de {col} según Churn")
    plt.xlabel("Churn")
    plt.ylabel(col)
    plt.show()

En `tenure` la diferencia es muy clara: los clientes con churn tienen mediana de ~10 meses frente a ~38 meses de los que se quedan. En `MonthlyCharges` los que hacen churn tienen medianas más altas (~$79 vs ~$64). `TotalCharges` muestra la diferencia inversa porque los clientes con churn llevan menos tiempo acumulando cargos.

In [ ]:
df.groupby("Churn")[numeric_analysis_cols].agg(["mean", "median", "std", "min", "max"]).round(2)

Los clientes con churn tienen una antigüedad media de 17.98 meses frente a los 37.57 de los que permanecen —los clientes nuevos son claramente más propensos a abandonar. En cargos mensuales, quienes hacen churn pagan en promedio $74.44 contra $61.27, lo que apunta a que los planes más caros generan más insatisfacción. Los cargos totales son menores para los clientes con churn ($1.531 vs $2.549) precisamente porque llevan menos tiempo en la empresa, no porque paguen menos por mes.

## Tasa de churn por categorías

A continuación se calcula la tasa de churn para las principales variables categóricas. Esta tasa indica el porcentaje de clientes que abandonaron la empresa dentro de cada categoría.

In [ ]:
def churn_rate_table(data, column):
    resumen = (
        data.groupby(column)
        .agg(
            clientes=(column, "count"),
            clientes_churn=("Churn", lambda x: (x == "Yes").sum()),
            churn_rate=("Churn", lambda x: (x == "Yes").mean() * 100)
        )
        .reset_index()
        .sort_values("churn_rate", ascending=False)
    )
    
    resumen["churn_rate"] = resumen["churn_rate"].round(2)
    return resumen

La función `churn_rate_table()` devuelve para cada categoría el total de clientes, cuántos hicieron churn y el porcentaje. Las filas se ordenan de mayor a menor tasa de churn para identificar fácilmente los grupos más riesgosos.

In [ ]:
variables_categoricas_clave = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "OnlineSecurity",
    "TechSupport",
    "PaperlessBilling",
    "OnlineBackup",
    "DeviceProtection"
]

for col in variables_categoricas_clave:
    print(f"\nTasa de churn por {col}")
    display(churn_rate_table(df, col))

Los patrones más destacados en las tablas: `PaymentMethod Electronic check` tiene el mayor churn con 45.29%, `Contract Month-to-month` con 42.71%, y los clientes sin `OnlineSecurity` (41.77%) o sin `TechSupport` (41.64%) también presentan tasas muy altas. En contraste, los clientes sin ningún servicio de internet tienen tasas de apenas 7.40%.

In [ ]:
for col in variables_categoricas_clave:
    resumen = churn_rate_table(df, col)
    
    plt.figure(figsize=(8, 4))
    sns.barplot(data=resumen, x=col, y="churn_rate")
    
    plt.title(f"Tasa de churn por {col}")
    plt.xlabel(col)
    plt.ylabel("Tasa de churn (%)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

Los gráficos ordenados de mayor a menor tasa muestran visualmente cuáles categorías son más riesgosas. `Electronic check` en PaymentMethod y `Month-to-month` en Contract sobresalen claramente por encima del resto.

## Análisis cruzado de variables categóricas

Además de analizar cada variable por separado, es útil estudiar combinaciones de variables. En este caso, se analiza la tasa de churn según el tipo de contrato y el método de pago.

In [ ]:
tabla_contract_payment = pd.crosstab(
    index=df["Contract"],
    columns=df["PaymentMethod"],
    values=(df["Churn"] == "Yes"),
    aggfunc="mean"
) * 100

tabla_contract_payment = tabla_contract_payment.round(2)

tabla_contract_payment

La combinación más riesgosa es contrato mensual + pago por cheque electrónico, con un 53.73% de churn. En el otro extremo, los contratos bianuales tienen tasas por debajo del 8% sin importar el método de pago.

In [ ]:
plt.figure(figsize=(10, 5))

sns.heatmap(
    tabla_contract_payment,
    annot=True,
    fmt=".2f",
    cmap="Reds"
)

plt.title("Tasa de churn (%) por tipo de contrato y método de pago")
plt.xlabel("Método de pago")
plt.ylabel("Tipo de contrato")
plt.show()

La celda más oscura del mapa es `Month-to-month / Electronic check` (53.73%), confirmando visualmente que es el perfil de mayor riesgo. Los contratos bianuales forman una fila casi completamente blanca en la parte inferior.

In [ ]:
tabla_contract_internet = pd.crosstab(
    index=df["Contract"],
    columns=df["InternetService"],
    values=(df["Churn"] == "Yes"),
    aggfunc="mean"
) * 100

tabla_contract_internet = tabla_contract_internet.round(2)

tabla_contract_internet

La combinación contrato mensual + fibra óptica es la de mayor churn con 54.61%. Los contratos anuales y bianuales reducen drásticamente esa tasa incluso para clientes con fibra óptica.

In [ ]:
plt.figure(figsize=(8, 5))

sns.heatmap(
    tabla_contract_internet,
    annot=True,
    fmt=".2f",
    cmap="Reds"
)

plt.title("Tasa de churn (%) por tipo de contrato y servicio de internet")
plt.xlabel("Servicio de internet")
plt.ylabel("Tipo de contrato")
plt.show()

La celda más oscura es `Month-to-month / Fiber optic` con 54.61%. Los contratos de largo plazo tienen tonos muy claros en todas las columnas, mostrando que el tipo de contrato tiene más influencia que el tipo de internet.

In [ ]:
PROCESSED_PATH = BASE_DIR / "data" / "processed" / "telco_churn_clean.csv"

df.to_csv(PROCESSED_PATH, index=False)

print(f"Dataset limpio guardado en: {PROCESSED_PATH}")

El dataset limpio (con `TotalCharges` corregida a `float64`) se guarda en `data/processed/` para usarse como entrada en el Notebook 2.

## Conclusiones principales del EDA

A partir del análisis exploratorio realizado, se identifican las siguientes conclusiones:

1. El dataset contiene 7.043 clientes y 21 variables, incluyendo información demográfica, servicios contratados, facturación y la variable objetivo `Churn`.

2. La variable objetivo presenta un desbalance moderado: aproximadamente el 73,46% de los clientes no abandonaron la empresa, mientras que el 26,54% sí lo hicieron. Por esta razón, en la etapa de modelado no será suficiente evaluar únicamente el `accuracy`.

3. La variable `tenure` muestra una relación importante con el churn. Los clientes que abandonaron tienen una permanencia promedio menor que los clientes que no abandonaron, lo cual indica que los clientes nuevos o con menor antigüedad pueden representar un grupo de mayor riesgo.

4. Los clientes con contrato mensual presentan una tasa de churn considerablemente más alta que los clientes con contratos de uno o dos años. Esto sugiere que la estabilidad contractual está relacionada con una menor probabilidad de abandono.

5. El método de pago también muestra diferencias relevantes. Los clientes que pagan mediante `Electronic check` presentan una tasa de churn más alta que aquellos que utilizan métodos automáticos como tarjeta de crédito o transferencia bancaria.

6. Las variables relacionadas con servicios adicionales, como `OnlineSecurity`, `TechSupport`, `OnlineBackup` y `DeviceProtection`, muestran diferencias importantes en la tasa de churn. En general, los clientes que no cuentan con estos servicios presentan mayor probabilidad de abandono.

7. Los clientes con servicio de internet por fibra óptica presentan una mayor tasa de churn frente a otros tipos de servicio. Este patrón debe ser analizado con cuidado en el modelado, ya que puede estar relacionado con costos, calidad percibida o características del segmento de clientes.

8. El análisis cruzado entre `Contract`, `PaymentMethod` e `InternetService` permite identificar perfiles específicos de mayor riesgo, especialmente clientes con contrato mensual y ciertos métodos de pago.

9. Para la etapa de modelado será importante usar métricas como `precision`, `recall`, `F1-score` y `AUC`, además de la matriz de confusión y curvas ROC.

10. Las variables identificadas en este EDA servirán como base para construir modelos de clasificación mediante `Pipeline`, `GridSearchCV` y algoritmos como Random Forest, XGBoost, CatBoost y LightGBM.